# 03. lc0 Evaluation Pipeline (Track B)

## CS377 Reinforcement Learning — Team 1
### Concentration vs. Dispersion in Free-Setup Chess via AlphaZero

---

This notebook contains the **lc0-based evaluation pipeline** (Track B):

1. **Lc0Runner** — UCI 프로토콜 기반 lc0 self-play 엔진
2. **run_lc0.py** — lc0 실험 실행 CLI
3. **run_arena.py** — Arena 평가 CLI

Track B는 프로젝트의 **주요 과학적 결과**를 생산하는 "strong reference" 트랙입니다.  
사전학습된 lc0 네트워크(~2200+ Elo)를 사용하여 핸디캡 포지션에서 self-play를 수행합니다.

#### 실험 모드:
- **Deterministic**: 매 수마다 n=800 노드 탐색 후 최선의 수 선택
- **Stochastic**: 첫 N 수는 MultiPV softmax 샘플링으로 다양한 오프닝 탐색

---
## 1. Lc0Runner — UCI Self-Play Engine

lc0 엔진을 UCI 프로토콜로 제어하여 핸디캡 포지션에서 self-play 게임을 실행합니다.

### Key Features:
- **Fixed-node search**: 강도 제어를 위해 nodes=800 고정
- **MultiPV softmax sampling**: 오프닝 다양성을 위한 stochastic 모드
- **Color-balanced evaluation**: 각 패턴을 NoQ=White, NoQ=Black 양쪽으로 테스트
- **JSONL logging**: `GameRecord` 형식으로 모든 결과 기록

Source: `handichess/track_b/lc0_runner.py`

In [ ]:
"""
lc0 game runner — Track B pipeline.

Runs self-play games using the lc0 engine (via UCI protocol) from
handicap starting positions. This is the "strong reference" track
that produces the main scientific results.

Requires:
  - lc0 binary installed and accessible
  - A neural network weights file (.pb.gz)
"""

from __future__ import annotations

import logging
import math
import random
import time
from pathlib import Path
from typing import Optional

import chess
import chess.engine
import chess.pgn

logger = logging.getLogger(__name__)


class Lc0Runner:
    """
    Runs lc0 self-play games from handicap positions.

    Two lc0 instances play against each other (or one instance alternates),
    with fixed nodes per move for strength control.
    """

    def __init__(
        self,
        engine_path: str = "lc0",
        weights_path: str = "",
        nodes: int = 800,
        threads: int = 1,
        backend: str = "cpu",
        exploration_plies: int = 16,
        temperature: float = 1.0,
        stochastic_plies: int = 0,
        multipv: int = 1,
        score_temperature_cp: float = 120.0,
        seed: int = 42,
    ):
        self.engine_path = engine_path
        self.weights_path = weights_path
        self.nodes = nodes
        self.threads = threads
        self.backend = backend
        self.exploration_plies = exploration_plies
        self.temperature = temperature
        self.stochastic_plies = stochastic_plies
        self.multipv = multipv
        self.score_temperature_cp = score_temperature_cp
        self.seed = seed
        self.rng = random.Random(seed)

    def _start_engine(self) -> chess.engine.SimpleEngine:
        """Start an lc0 UCI engine instance."""
        engine = chess.engine.SimpleEngine.popen_uci(self.engine_path)

        options = {
            "Threads": self.threads,
            "Backend": self.backend,
            "Temperature": 1.0,
            "TempDecayMoves": 20,
        }
        if self.weights_path:
            options["WeightsFile"] = self.weights_path

        for key, value in options.items():
            try:
                engine.configure({key: value})
            except chess.engine.EngineError:
                logger.warning(f"Could not set engine option: {key}={value}")

        return engine

    def _sample_multipv_move(
        self,
        engine: chess.engine.SimpleEngine,
        board: chess.Board,
    ) -> chess.Move:
        """Sample from LC0's top MultiPV moves using a softmax over centipawns."""
        infos = engine.analyse(
            board,
            chess.engine.Limit(nodes=self.nodes),
            multipv=self.multipv,
        )
        if isinstance(infos, dict):
            infos = [infos]

        candidates = []
        for info in infos:
            pv = info.get("pv") or []
            if not pv:
                continue
            move = pv[0]
            if move not in board.legal_moves:
                continue
            score = info.get("score")
            cp = score.pov(board.turn).score(mate_score=10000) if score else 0
            candidates.append((move, cp))

        if not candidates:
            result = engine.play(board, chess.engine.Limit(nodes=self.nodes))
            return result.move

        # Softmax sampling over centipawn scores
        max_cp = max(cp for _, cp in candidates)
        temp = max(self.score_temperature_cp, 1e-6)
        weights = [math.exp((cp - max_cp) / temp) for _, cp in candidates]
        total = sum(weights)
        pick = self.rng.random() * total
        acc = 0.0
        for (move, _), weight in zip(candidates, weights):
            acc += weight
            if pick <= acc:
                return move
        return candidates[-1][0]

    def play_game(
        self,
        start_fen: str,
        pattern_id: str,
        q_side: str,
    ) -> "GameRecord":
        """
        Play a single game from a match-up position.
        Both sides are played by the same lc0 instance (self-play).
        """
        engine = self._start_engine()

        try:
            board = chess.Board(start_fen)
            ply = 0

            while not board.is_game_over(claim_draw=True):
                if ply >= 512:
                    break

                if self.stochastic_plies > 0 and ply < self.stochastic_plies and self.multipv > 1:
                    move = self._sample_multipv_move(engine, board)
                else:
                    result = engine.play(board, chess.engine.Limit(nodes=self.nodes))
                    move = result.move
                if move is None:
                    break

                board.push(move)
                ply += 1

            # Determine outcome
            outcome = board.outcome(claim_draw=True)
            if outcome is None:
                outcome_str = "1/2-1/2"
                termination = "max_moves"
            else:
                if outcome.winner is None:
                    outcome_str = "1/2-1/2"
                elif outcome.winner == chess.WHITE:
                    outcome_str = "1-0"
                else:
                    outcome_str = "0-1"
                termination = outcome.termination.name.lower()

            result_str, result_score = result_from_outcome(outcome_str, q_side)

            # Create PGN string
            pgn_game = chess.pgn.Game.from_board(board)
            pgn_game.headers["Event"] = f"Track B Evaluation ({pattern_id})"
            pgn_game.headers["White"] = "Lc0" if q_side == "white" else "Lc0 (NoQ)"
            pgn_game.headers["Black"] = "Lc0" if q_side == "black" else "Lc0 (NoQ)"
            pgn_game.headers["Result"] = outcome_str

            return GameRecord(
                pattern_id=pattern_id,
                q_side=q_side,
                noq_side="black" if q_side == "white" else "white",
                result=result_str,
                result_score=result_score,
                ply=ply,
                start_fen=start_fen,
                termination=termination,
                engine=f"lc0_n{self.nodes}",
                nodes=self.nodes,
                extra={
                    "pgn": str(pgn_game),
                    "sampling": (
                        f"multipv_softmax_first_{self.stochastic_plies}_plies"
                        if self.stochastic_plies > 0 and self.multipv > 1
                        else "deterministic"
                    ),
                    "multipv": self.multipv,
                    "score_temperature_cp": self.score_temperature_cp,
                    "seed": self.seed,
                }
            )

        finally:
            engine.quit()

    def run_pattern(
        self,
        pattern_id: str,
        num_games: int,
        log: "GameLog",
    ) -> dict:
        """Run multiple games for a single pattern, color-balanced."""
        pattern = get_pattern_by_id(pattern_id)
        games_per_side = num_games // 2

        wins = draws = losses = 0

        for noq_color in ["white", "black"]:
            pos = generate_position(pattern, chess.WHITE if noq_color == "white" else chess.BLACK)
            q_side = "black" if noq_color == "white" else "white"

            for g in range(games_per_side):
                logger.info(
                    f"Game {g+1}/{games_per_side} | "
                    f"pattern={pattern_id} | noq_color={noq_color} (q_side={q_side})"
                )

                record = self.play_game(pos.fen, pattern_id, q_side)
                log.write(record)

                if record.result == "win":
                    wins += 1
                elif record.result == "draw":
                    draws += 1
                else:
                    losses += 1

                logger.info(
                    f"  Result: {record.result} in {record.ply} ply "
                    f"({record.termination})"
                )

        total = wins + draws + losses
        stats = {
            "pattern_id": pattern_id,
            "wins": wins, "draws": draws, "losses": losses,
            "score": (wins + 0.5 * draws) / max(total, 1),
            "total": total,
        }
        logger.info(f"Pattern {pattern_id} complete: {wins}W/{draws}D/{losses}L = {stats['score']:.3f}")
        return stats

    def run_all_patterns(
        self,
        num_games_per_pattern: int,
        log_path: str,
        phase: Optional[int] = None,
    ) -> list[dict]:
        """Run games for all patterns (or a specific phase)."""
        log = GameLog(log_path)
        patterns = get_patterns(phase=phase)
        all_stats = []

        for pattern in patterns:
            stats = self.run_pattern(
                pattern.pattern_id,
                num_games_per_pattern,
                log,
            )
            all_stats.append(stats)

        return all_stats

    def evaluate_position(self, fen: str, nodes: Optional[int] = None) -> dict:
        """Get lc0's evaluation of a position (for sanity checking)."""
        engine = self._start_engine()
        try:
            board = chess.Board(fen)
            limit = chess.engine.Limit(nodes=nodes or self.nodes)
            info = engine.analyse(board, limit)

            score = info.get("score")
            pv = info.get("pv", [])

            return {
                "fen": fen,
                "score_cp": score.white().score(mate_score=10000) if score else None,
                "score_mate": score.white().mate() if score and score.white().mate() is not None else None,
                "pv": [m.uci() for m in pv[:5]],
                "depth": info.get("depth"),
                "nodes": info.get("nodes"),
            }
        finally:
            engine.quit()

---
## 2. Experiment Execution Scripts

### run_lc0.py — lc0 실험 실행 CLI

```bash
# Deterministic (default)
python scripts/run_lc0.py --pattern rook_bishop_pawn --games 100 --nodes 800

# Stochastic (MultiPV softmax for first 20 plies)
python scripts/run_lc0.py --pattern rook_bishop_pawn --games 100 --nodes 800 \
    --stochastic-plies 20 --multipv 5 --score-temperature-cp 120
```

Source: `scripts/run_lc0.py`

In [ ]:
"""Run lc0 self-play games from handicap positions (Track B)."""

import argparse
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


def main_run_lc0():
    parser = argparse.ArgumentParser(description="Run lc0 handicap games")
    parser.add_argument("--pattern", type=str, default=None,
                        help="Specific pattern ID (default: all Phase 1)")
    parser.add_argument("--games", type=int, default=100,
                        help="Games per pattern")
    parser.add_argument("--nodes", type=int, default=800,
                        help="Nodes per move")
    parser.add_argument("--engine", "--lc0-path", dest="engine", type=str, default="lc0",
                        help="Path to lc0 binary")
    parser.add_argument("--weights", "--weights-path", dest="weights", type=str, default="",
                        help="Path to lc0 weights file")
    parser.add_argument("--backend", type=str, default="cuda-auto",
                        help="Lc0 neural network backend")
    parser.add_argument("--threads", type=int, default=1,
                        help="Lc0 CPU worker threads")
    parser.add_argument("--stochastic-plies", type=int, default=0,
                        help="Use MultiPV sampling for the first N half-moves")
    parser.add_argument("--multipv", type=int, default=1,
                        help="Number of LC0 principal variations to sample from")
    parser.add_argument("--score-temperature-cp", type=float, default=120.0,
                        help="Softmax temperature in centipawns for MultiPV sampling")
    parser.add_argument("--seed", type=int, default=42,
                        help="Random seed for stochastic move sampling")
    parser.add_argument("--output", type=str, default="runs/lc0_games.jsonl",
                        help="Output log file")
    parser.add_argument("--phase", type=int, default=1,
                        help="Phase filter (1 or 2)")
    args = parser.parse_args()

    runner = Lc0Runner(
        engine_path=args.engine,
        weights_path=args.weights,
        nodes=args.nodes,
        threads=args.threads,
        backend=args.backend,
        stochastic_plies=args.stochastic_plies,
        multipv=args.multipv,
        score_temperature_cp=args.score_temperature_cp,
        seed=args.seed,
    )

    if args.pattern:
        log = GameLog(args.output)
        runner.run_pattern(args.pattern, args.games, log)
    else:
        runner.run_all_patterns(args.games, args.output, phase=args.phase)


print("See scripts/run_lc0.py for the full CLI script.")
print("Example: python scripts/run_lc0.py --pattern rook_bishop_pawn --games 100 --nodes 800")

---
### run_arena.py — Arena 평가 CLI

```bash
# Evaluate trained model vs random baseline
python scripts/run_arena.py --game chess --checkpoint runs/checkpoints/final.pt \
    --baseline random --games 40 --pattern rook_bishop_pawn
```

Source: `scripts/run_arena.py`

In [ ]:
"""Run arena evaluation between two checkpoints or against a baseline."""

import argparse
import json
import logging
from pathlib import Path

import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


def main_run_arena():
    parser = argparse.ArgumentParser(description="Run arena evaluation")
    parser.add_argument("--game", type=str, default="tictactoe",
                        choices=["tictactoe", "chess"])
    parser.add_argument("--checkpoint", type=str, required=True,
                        help="Path to model checkpoint")
    parser.add_argument("--baseline", type=str, default="random",
                        choices=["random", "greedy", "weak_mcts", "self"],
                        help="Baseline type")
    parser.add_argument("--games", type=int, default=40,
                        help="Number of games")
    parser.add_argument("--pattern", type=str, default=None,
                        help="Handicap pattern (chess only)")
    parser.add_argument("--noq-color", type=str, default="white",
                        choices=["white", "black"])
    parser.add_argument("--device", type=str, default="cpu")
    parser.add_argument("--num-simulations", type=int, default=200)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output", type=str, default=None)
    args = parser.parse_args()

    # Create game, load net, and evaluate
    # (See scripts/run_arena.py for full implementation)
    print(f"Would run: {args.game} vs {args.baseline}, {args.games} games")


print("See scripts/run_arena.py for the full CLI script.")
print("Example: python scripts/run_arena.py --game chess --checkpoint runs/checkpoints/final.pt --baseline random")